# 第二週完整講義 B：Python世界時鐘

以原世界時鐘的城市名稱串列、固定時差函式、for走訪、while更新及日夜判斷為主線，將日期時間問題拆成可以觀察、修改及測試的小步驟。

本版先以固定UTC時刻講授運算與流程，再用城市時區處理夏令時間。原始兩個無窮迴圈改為有限次更新示範，原DataFrame及日夜判斷功能保留為整合段。原始檔另存於來源資料夾的「第二週範例原始備份_20261022」。

## 第二週共同教學路線

這兩份筆記本合起來是一堂180分鐘的課程，含10分鐘休息。第一週已學變數與基本輸出；本週以兩個作品逐步補上運算、選擇、重複、函式及串列。函式與串列先作必要的入門，完整練習可在課後繼續。

| 時間 | 講授與操作 | 範例對應 |
|---|---|---|
| 0–10分 | 執行環境、輸入與輸出、作品拆解 | 兩份的第0節 |
| 10–35分 | 運算元、型別、優先順序、比較與邏輯 | 計算機C1、世界時鐘T1 |
| 35–55分 | if/elif/else、條件式運算、邊界值 | 計算機C2、世界時鐘T2 |
| 55–80分 | def、參數、return、區域變數 | 計算機C3、世界時鐘T3 |
| 80–90分 | 休息 | |
| 90–115分 | 一維與二維串列、索引、走訪 | 計算機C4、世界時鐘T4 |
| 115–145分 | for、range、while、break、continue | 計算機C5–C6、世界時鐘T5–T6 |
| 145–165分 | 對照原程式、整合互動操作 | 計算機C7–C8、世界時鐘T7–T8 |
| 165–180分 | 修改題、邊界測試與口頭說明 | 兩份第9節擇一實作，另一份課後完成 |

講課時先請學員預測輸出，再執行，最後改一個值重跑。每節的「講授」可直接作為口語說明；「檢核」用來確認是否理解。從頭執行程式儲存格即可重建資料，所有自動示範都會結束，不需要先輸入資料。

### 補充投影片對照

頁碼是PPTX中由第一張開始計算的投影片序號，包含封面及空白頁。下列範圍對應原投影片內容；函式與陣列的完整教學由本筆記本新增。

| 補充投影片 | 頁碼與主題 | 計算機位置 | 世界時鐘位置 |
|---|---|---|---|
| Py01_基本認識(1).pptx | 8–18命名、20–34型別與指派、42–47轉型 | C0–C1、C3 | T0–T3 |
| Py01_基本認識(2).pptx | 5–8輸出、24–26 f-string、29–37輸入與eval | C1、C7–C8 | T1、T3、T7 |
| Py01_基本認識(3).pptx | 3–18算術與優先順序、22–28比較邏輯、32–36位元運算 | C1–C2 | T1–T2 |
| Py02_選擇敘述與迴圈(1).pptx | 3–5流程、9–16縮排、24–28雙向判斷、32–38多向與巢狀判斷 | C2–C3、C8 | T2、T6 |
| Py02_選擇敘述與迴圈(2).pptx | 6–11 while、14–22 for/range、26–30 break/continue、32–38迴圈else與巢狀迴圈 | C4–C6 | T4–T6 |

### 講授時的精確用語

`//` 是向下取整，負數不能解釋成單純刪掉小數：`-7 // 2 == -4`。`and`、`or` 有短路行為，不能一般性地用 `&`、`|` 取代；後兩者在整數上是位元運算。這兩點分別補充 Py01(3)第13、27頁。`=` 是指派，`==` 才是相等比較。字串中的 `%H` 是時間格式代碼，與數值的 `%` 餘數運算分屬不同語境。

本課用 Python 的 `list`（串列）承接「陣列」的入門概念；串列可以混合型別、改變長度，與 NumPy 的同質數值陣列不同。原題兩份程式均以串列為主，不要求先安裝NumPy。

## T0 作品拆解與可重現資料

**講授：** 世界時鐘不是每個城市各有一個獨立的現在。所有城市應從同一個UTC時刻轉換，差別在顯示的當地日期與時間。原程式在每次呼叫timezone時各自取現在，可能剛好跨秒；新版先取一次，再交給所有城市。

本課區分兩個模型：固定偏移量供練習加減、整除與餘數；城市時區供實際顯示，會根據日期處理夏令時間。倫敦+1、紐約-4、雪梨+11都不是全年固定值。原範例的時差數字只能視為特定日期的假設。

先固定UTC為2026-01-15 20:30，避免每次輸出不同而難以核對。基本段只用標準函式庫；最後的表格需pandas，城市時區需作業系統時區資料或tzdata套件，缺少時依訊息安裝後重跑。

In [1]:
from datetime import datetime, timedelta, timezone
import time
import html
from IPython.display import display, HTML

BASE_UTC = datetime(2026, 1, 15, 20, 30, 0, tzinfo=timezone.utc)
print("統一基準時刻：", BASE_UTC.isoformat())

統一基準時刻： 2026-01-15T20:30:00+00:00


## T1 時間運算與跨日

**講授：** 先把20:30轉成分鐘：`20*60+30=1230`。台灣固定偏移+8小時，即加480分鐘，得到1710。一天有1440分鐘，`1710//1440`得到跨越的天數1，`1710%1440`得到當日的270分鐘；`270//60`是4點，`270%60`是30分。

負偏移也適用：UTC 01:15減4小時是-165分鐘，`-165//1440`為-1，代表前一天；餘數1275對應21:15。這說明整除必須向下取整。僅用`(hour+offset)%24`只能取得小時，會丟掉日期資訊。

偏移可能含半小時，因此資料以分鐘保存，例如+5:30是330分鐘，不能四捨五入為5或6小時。f-string的`:02d`將整數補成兩位；datetime的`strftime("%H:%M:%S")`是另一種格式化方式。

**檢核：** UTC 20:30加9小時應為隔日05:30；UTC 01:15減4小時應為前日21:15。下例先使用手算模型，再由datetime負責真正的年月日變動。

In [2]:
utc_minutes = 20 * 60 + 30
offset_minutes = 8 * 60
total_minutes = utc_minutes + offset_minutes
day_shift = total_minutes // (24 * 60)
minute_of_day = total_minutes % (24 * 60)
hour = minute_of_day // 60
minute = minute_of_day % 60
print(f"跨日數={day_shift}，當地時間={hour:02d}:{minute:02d}")

previous_day_total = 1 * 60 + 15 - 4 * 60
print("前一天測試：", previous_day_total // 1440,
      f"{previous_day_total % 1440 // 60:02d}:{previous_day_total % 60:02d}")
local_dt = BASE_UTC.astimezone(timezone(timedelta(hours=8)))
print("實際日期：", local_dt.strftime("%Y-%m-%d %H:%M:%S"))

跨日數=1，當地時間=04:30
前一天測試： -1 21:15
實際日期： 2026-01-16 04:30:00


## T2 判斷式與日夜邊界

**講授：** 本教學把06:00至17:59:59定義為白天，其餘晚上，這是介面標籤規則，並未計算各地實際日出日落。`6 <= hour < 18`等價於`hour >= 6 and hour < 18`。18點是晚上，不能一邊寫「到18點」一邊又把18包含在白天。

先排除不合法時數，再比較範圍，是清楚的多向判斷。若只分日夜可用if/else；若增加凌晨、上午、下午、夜晚，則用elif。不要寫`hour >= 6 or hour < 18`，合法小時幾乎全部會成立。

**邊界表：** 05:59為晚上，06:00為白天，17:59為白天，18:00為晚上。這四筆比只測12:00更容易發現錯誤。原程式`judge_day_night`與這一節對應。

In [3]:
for hour in [5, 6, 17, 18, 23, 24]:
    if not 0 <= hour < 24:
        label = "無效小時"
    elif 6 <= hour < 18:
        label = "白天"
    else:
        label = "晚上"
    print(hour, label)
print("正確and：", 5 >= 6 and 5 < 18)
print("誤用or：", 5 >= 6 or 5 < 18)

5 晚上
6 白天
17 白天
18 晚上
23 晚上
24 無效小時
正確and： False
誤用or： True


## T3 函式、參數與回傳值

**講授：** 原`timezone(h)`將取時間與格式化寫在一起。新版`fixed_local(base_utc, offset_minutes)`接受明確基準與分鐘偏移，回傳datetime；`judge_day_night`回傳文字標籤。這樣可先測資料，再決定印文字、畫HTML或建DataFrame。

定義函式不會立即執行。每次呼叫都有自己的區域變數；return將結果交回，離開後不會繼續執行函式剩下的敘述。`base_utc`具時區資訊稱為aware datetime；沒有時區資訊時無法確定是哪個時刻，本版主動提出錯誤。

原範例函式名稱timezone與標準庫同名，容易混淆，本版改稱fixed_local。`format_offset`示範條件式運算、abs、整除與格式化，能正確顯示UTC-03:30，而不是錯成UTC-03:-30。

**檢核：** 為何不在fixed_local內直接呼叫datetime.now？為了用指定時刻重現結果，並確保同一張表的城市共用基準。

In [4]:
def fixed_local(base_utc, offset_minutes):
    if base_utc.tzinfo is None or base_utc.utcoffset() is None:
        raise ValueError("基準時間必須含時區資訊")
    if type(offset_minutes) is not int or not -1440 < offset_minutes < 1440:
        raise ValueError("偏移分鐘必須為介於-1440與1440之間的整數")
    return base_utc.astimezone(timezone(timedelta(minutes=offset_minutes)))

def judge_day_night(time_str):
    parsed = datetime.strptime(time_str, "%H:%M:%S")
    if 6 <= parsed.hour < 18:
        return "白天"
    return "晚上"

def format_offset(offset_minutes):
    sign = "+" if offset_minutes >= 0 else "-"
    minutes = abs(offset_minutes)
    return f"UTC{sign}{minutes // 60:02d}:{minutes % 60:02d}"

print(fixed_local(BASE_UTC, 480))
print(format_offset(330), format_offset(-210))
print(judge_day_night("18:00:00"))

2026-01-16 04:30:00+08:00
UTC+05:30 UTC-03:30
晚上


## T4 城市陣列、索引與資料配對

**講授：** 原name與loc_time是平行串列，同一個索引代表同一城市。這種方法容易懂，但必須維持長度及順序一致；只增加城市而沒有增加偏移，就會出錯。`range(4)`也把筆數寫死，改成range(len(name))才能隨資料擴充。

一維串列示範索引、切片及append。下一步把城市和偏移放在同一筆紀錄，形成「串列中的字典」，迴圈讀一筆就能取得所有欄位。這是資料結構延伸，主要目的是避免資料錯配。若用兩欄二維串列也能表達，但字典的欄位名稱更清楚。

Python list可放文字和數字，並不是固定型別的數值陣列。若未來改用NumPy，數值陣列乘2可逐項倍增，而Python串列乘2會重複內容。本題不需要NumPy即可完成。

In [5]:
names = ["台灣", "日本", "印度"]
offsets = [480, 540, 330]  # 分鐘；本段明確採固定偏移
assert len(names) == len(offsets)
print("第一座：", names[0], offsets[0])
print("前兩座：", names[:2], "最後一座：", names[-1])
print("串列乘2：", [480, 540] * 2)

fixed_cities = [
    {"city": "台灣", "offset_minutes": 480},
    {"city": "日本", "offset_minutes": 540},
    {"city": "印度", "offset_minutes": 330},
]
fixed_cities.append({"city": "UTC示範站", "offset_minutes": 0})
print("目前城市數：", len(fixed_cities))

第一座： 台灣 480
前兩座： ['台灣', '日本'] 最後一座： 印度
串列乘2： [480, 540, 480, 540]
目前城市數： 4


## T5 for走訪與表格資料

**講授：** 對每座城市做一樣的轉換，選for最自然。`enumerate`同時產生索引與城市；若使用平行串列，可先檢查長度後再用zip配對。zip預設遇到最短串列就停止，因此不能靠它發現少了一筆資料。

下例每輪先計算local，取日期與時間，呼叫日夜判斷，再append一筆字典。輸出格式不必參與計算，rows保留為可再加工的資料。`continue`可略過不需要顯示的城市；搜尋時用break提早停止，迴圈else可表示全部找完仍未發現目標。

**追蹤：** BASE_UTC是1月15日20:30，所以台灣為1月16日04:30、日本05:30、印度02:00，皆標為晚上。請先預測，再查看輸出。

In [6]:
def build_fixed_rows(base_utc, cities):
    rows = []
    for item in cities:
        local = fixed_local(base_utc, item["offset_minutes"])
        clock_text = local.strftime("%H:%M:%S")
        rows.append({"城市": item["city"], "時區": format_offset(item["offset_minutes"]),
                     "日期": local.strftime("%Y-%m-%d"), "時間": clock_text,
                     "狀態": judge_day_night(clock_text)})
    return rows

fixed_rows = build_fixed_rows(BASE_UTC, fixed_cities)
for index, row in enumerate(fixed_rows, start=1):
    print(index, row)

for row in fixed_rows:
    if row["城市"] == "日本":
        print("找到日本：", row["時間"])
        break
else:
    print("查無城市")
print("只顯示亞洲三座城市：")
for row in fixed_rows:
    if row["城市"] == "UTC示範站":
        continue
    print(row["城市"], row["時間"])

1 {'城市': '台灣', '時區': 'UTC+08:00', '日期': '2026-01-16', '時間': '04:30:00', '狀態': '晚上'}
2 {'城市': '日本', '時區': 'UTC+09:00', '日期': '2026-01-16', '時間': '05:30:00', '狀態': '晚上'}
3 {'城市': '印度', '時區': 'UTC+05:30', '日期': '2026-01-16', '時間': '02:00:00', '狀態': '晚上'}
4 {'城市': 'UTC示範站', '時區': 'UTC+00:00', '日期': '2026-01-15', '時間': '20:30:00', '狀態': '晚上'}
找到日本： 05:30:00
只顯示亞洲三座城市：
台灣 04:30:00
日本 05:30:00
印度 02:00:00


## T6 while更新、計數與結束

**講授：** for回答「這一輪要處理哪些城市」，while回答「要不要再做下一輪」。巢狀後，每更新一次就逐一處理所有城市。原while True一直執行，後面的Notebook儲存格必須等中斷後才會開始。

教學時先用有限次更新：refresh_count從0開始，條件為小於max_refresh，每輪加1。示範把基準時刻每輪加1秒，結果可重現；這是模擬走時，不是作業系統的真實時鐘。實際版在T8每輪重新取UTC現在。

`time.sleep(1)`只暫停程式，不保證畫面精準每秒刷新，整輪還有計算及顯示耗時。Notebook互動介面中，長時間阻塞也會妨礙其他操作；本版預設只跑3次。若手動中斷則捕捉KeyboardInterrupt，保留已完成結果。

**檢核：** max_refresh=0應完全不更新；如果忘記refresh_count+=1會發生什麼？while條件一直為真，形成無窮迴圈。

In [7]:
refresh_count = 0
max_refresh = 3
while refresh_count < max_refresh:
    simulated_utc = BASE_UTC + timedelta(seconds=refresh_count)
    rows = build_fixed_rows(simulated_utc, fixed_cities)
    print(f"第{refresh_count + 1}輪", [(r["城市"], r["時間"]) for r in rows])
    refresh_count += 1
print("模擬更新完成，共", refresh_count, "輪")

第1輪 [('台灣', '04:30:00'), ('日本', '05:30:00'), ('印度', '02:00:00'), ('UTC示範站', '20:30:00')]
第2輪 [('台灣', '04:30:01'), ('日本', '05:30:01'), ('印度', '02:00:01'), ('UTC示範站', '20:30:01')]
第3輪 [('台灣', '04:30:02'), ('日本', '05:30:02'), ('印度', '02:00:02'), ('UTC示範站', '20:30:02')]
模擬更新完成，共 3 輪


## T7 城市時區與夏令時間

**講授：** 真正的世界時鐘應保存`Europe/London`、`America/New_York`等城市時區名稱，而非全年固定的+1或-4。Python的zoneinfo依時區資料庫及指定日期決定偏移。對同一UTC時刻執行astimezone即可取得對應城市的日期時間。

原範例的name與loc_time改為cities中的city與zone。仍然使用list、for、函式及if，新增工具只是取代「固定偏移」這一部分。冬夏比較使用2026年1月15日及7月15日的固定UTC時刻；這些日期是測試資料，不是現在時間。

若Windows缺少時區資料，安裝`%pip install tzdata`再重跑。套件資料可能隨各地規則更新，因此本節由本機時區資料庫實際換算，不自行推測某城市當下的偏移。

In [8]:
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

cities = [
    {"city": "倫敦", "zone": "Europe/London"},
    {"city": "台灣", "zone": "Asia/Taipei"},
    {"city": "日本", "zone": "Asia/Tokyo"},
    {"city": "紐約", "zone": "America/New_York"},
    {"city": "雪梨", "zone": "Australia/Sydney"},
    {"city": "巴黎", "zone": "Europe/Paris"},
]

def build_city_rows(base_utc, city_records):
    if base_utc.tzinfo is None or base_utc.utcoffset() is None:
        raise ValueError("基準時間必須含時區資訊")
    rows = []
    for item in city_records:
        local = base_utc.astimezone(ZoneInfo(item["zone"]))
        offset = int(local.utcoffset().total_seconds() // 60)
        clock_text = local.strftime("%H:%M:%S")
        rows.append({"城市": item["city"], "時區": item["zone"],
                     "UTC偏移": format_offset(offset), "日期": local.strftime("%Y-%m-%d"),
                     "時間": clock_text, "狀態": judge_day_night(clock_text)})
    return rows

ZONE_DATA_READY = True
try:
    for item in cities:
        ZoneInfo(item["zone"])
except ZoneInfoNotFoundError:
    ZONE_DATA_READY = False
    print("請安裝時區資料後重跑：%pip install tzdata")

if ZONE_DATA_READY:
    for month in [1, 7]:
        sample = datetime(2026, month, 15, 12, tzinfo=timezone.utc)
        print(sample.date(), [(r["城市"], r["UTC偏移"]) for r in build_city_rows(sample, cities)])

2026-01-15 [('倫敦', 'UTC+00:00'), ('台灣', 'UTC+08:00'), ('日本', 'UTC+09:00'), ('紐約', 'UTC-05:00'), ('雪梨', 'UTC+11:00'), ('巴黎', 'UTC+01:00')]
2026-07-15 [('倫敦', 'UTC+01:00'), ('台灣', 'UTC+08:00'), ('日本', 'UTC+09:00'), ('紐約', 'UTC-04:00'), ('雪梨', 'UTC+10:00'), ('巴黎', 'UTC+02:00')]


## T8 完整世界時鐘與DataFrame

**講授：** 整合順序為「取一次UTC現在、for轉換所有城市、產生HTML、更新顯示、累加輪數、暫停、檢查while條件」。HTML只是畫面文字，真正資料保存在rows裡。日夜顏色由if/else決定，文字插入HTML前用escape處理。

原版clear_output會清除儲存格的所有輸出，新版以display handle只更新該時鐘區域。max_refresh預設3，interval預設1秒；課堂示範縮短為0.2秒。要延長可呼叫`run_world_clock(max_refresh=10, interval=1)`，仍會自動結束。

下面DataFrame接續使用同一份rows。原`df.copy()`建立另一份表格；`apply(judge_day_night)`逐筆傳入時間字串，概念上等同for產生狀態串列。先理解for，再閱讀apply，避免只記住方法名稱。

In [9]:
def render_clock(rows):
    parts = ["<div style='font-family:sans-serif;max-width:700px'>"]
    for row in rows:
        color = "#a65b00" if row["狀態"] == "白天" else "#2458a6"
        parts.append(f"<p><b>{html.escape(row['城市'])}</b> "
                     f"{html.escape(row['日期'])} "
                     f"<span style='color:{color}'>{html.escape(row['時間'])}</span> "
                     f"{html.escape(row['UTC偏移'])} {html.escape(row['狀態'])}</p>")
    parts.append("</div>")
    return "".join(parts)

def run_world_clock(max_refresh=3, interval=1):
    if type(max_refresh) is not int or max_refresh < 0 or interval < 0:
        raise ValueError("更新次數須為非負整數，間隔不可為負")
    if not ZONE_DATA_READY:
        print("先安裝tzdata，再執行T7。")
        return []
    count = 0
    rows = []
    handle = None
    try:
        while count < max_refresh:
            base_utc = datetime.now(timezone.utc)
            rows = build_city_rows(base_utc, cities)
            content = HTML(render_clock(rows))
            if handle is None:
                handle = display(content, display_id=True)
            else:
                handle.update(content)
            count += 1
            if count < max_refresh:
                time.sleep(interval)
    except KeyboardInterrupt:
        print("使用者停止更新。")
    print(f"本次完成{count}輪更新。")
    return rows

latest_rows = run_world_clock(max_refresh=3, interval=0.2)

本次完成3輪更新。


In [10]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("表格延伸需pandas：%pip install pandas；固定時差與文字版不受影響。")

if pd is not None and latest_rows:
    df = pd.DataFrame(latest_rows)
    df_status = df.copy()
    df_status["狀態"] = df_status["時間"].apply(judge_day_night)
    display(df_status)

,城市,時區,UTC偏移,日期,時間,狀態
0,倫敦,Europe/London,UTC+01:00,2026-10-22,16:22:20,白天
1,台灣,Asia/Taipei,UTC+08:00,2026-10-22,23:22:20,晚上
2,日本,Asia/Tokyo,UTC+09:00,2026-10-23,00:22:20,晚上
3,紐約,America/New_York,UTC-04:00,2026-10-22,11:22:20,白天
4,雪梨,Australia/Sydney,UTC+11:00,2026-10-23,02:22:20,晚上
5,巴黎,Europe/Paris,UTC+02:00,2026-10-22,17:22:20,白天


## T9 隨堂練習與參考解答

1. 運算式：UTC 23:45加330分鐘，當地日期要加幾天，時間是多少？
2. 判斷式：將上班時段定為09:00至16:59，寫出函式並測試08:59、09:00、16:59、17:00。
3. 陣列與for：加入印度，城市時區採Asia/Kolkata，不要改迴圈次數或寫死第7筆。
4. while：讓實際時鐘更新5輪後停止，再改成0輪，說明輸出差異。
5. 函式與整合：輸入城市名稱查詢該列資料；找不到時回傳None，交由呼叫端顯示訊息。

**參考解答：** 第一題隔日05:15。第二題條件為`9 <= hour < 17`，四筆分別False、True、True、False。第三題在城市串列加入一筆字典即可。第四題將max_refresh設為5或0；零次不顯示時鐘、回傳空串列。第五題以for搜尋，找到便return，整個迴圈結束後return None。

**口頭評量：** 請學員指著自己的程式說出：哪個值控制重複？哪個分支決定顏色？日期跨日在哪裡處理？哪個函式可以不用畫面就先測？

In [11]:
def is_work_time(time_str):
    parsed = datetime.strptime(time_str, "%H:%M:%S")
    return 9 <= parsed.hour < 17

def find_city(rows, city_name):
    for row in rows:
        if row["城市"] == city_name:
            return row
    return None

exercise_cities = cities + [{"city": "印度", "zone": "Asia/Kolkata"}]
if ZONE_DATA_READY:
    exercise_rows = build_city_rows(BASE_UTC, exercise_cities)
    print("印度資料：", find_city(exercise_rows, "印度"))
    print("查無城市：", find_city(exercise_rows, "不存在"))
print("上班邊界：", [is_work_time(t) for t in ["08:59:00", "09:00:00", "16:59:00", "17:00:00"]])
# 延伸操作：run_world_clock(max_refresh=5, interval=1)

印度資料： {'城市': '印度', '時區': 'Asia/Kolkata', 'UTC偏移': 'UTC+05:30', '日期': '2026-01-16', '時間': '02:00:00', '狀態': '晚上'}
查無城市： None
上班邊界： [False, True, True, False]


## T10 測試與原程式對照

測試使用固定基準時間，所以不會隨現在時間改變。城市時區的季節測試只在資料庫可用時執行；未安裝時會清楚列出略過項目。

| 原範例位置 | 本版完整講授與程式 |
|---|---|
| timezone(h)、timedelta(hours=h) | T1、T3固定時差模型 |
| name與loc_time平行串列 | T4資料配對、T7城市時區 |
| for i in range(4)及range(len(name)) | T4、T5資料驅動走訪 |
| while True、sleep、KeyboardInterrupt | T6有限示範、T8有限實際更新 |
| strftime、split日期時間、HTML | T1、T3、T8 |
| dates、times、gmt_offsets、DataFrame | T5字典列、T8表格 |
| judge_day_night與apply | T2、T3、T8 |

**與計算機相互比較：** 計算機的二維串列描述畫面配置，時鐘串列保存城市資料；兩者都用for走訪。計算機由按鍵事件觸發一次計算，時鐘由有限while主動更新。兩者都能把核心函式抽出來，不靠視窗先驗證結果。

In [12]:
assert fixed_local(BASE_UTC, 480).strftime("%Y-%m-%d %H:%M") == "2026-01-16 04:30"
early = datetime(2026, 1, 15, 1, 15, tzinfo=timezone.utc)
assert fixed_local(early, -240).strftime("%Y-%m-%d %H:%M") == "2026-01-14 21:15"
assert format_offset(330) == "UTC+05:30"
assert format_offset(-210) == "UTC-03:30"
assert [judge_day_night(t) for t in ["05:59:59", "06:00:00", "17:59:59", "18:00:00"]] == ["晚上", "白天", "白天", "晚上"]
assert [is_work_time(t) for t in ["08:59:00", "09:00:00", "16:59:00", "17:00:00"]] == [False, True, True, False]
assert build_fixed_rows(BASE_UTC, []) == []
assert find_city(fixed_rows, "不存在") is None
assert run_world_clock(max_refresh=0, interval=0) == []
for bad_time in ["24:00:00", "12:60:00", "abc"]:
    try:
        judge_day_night(bad_time)
    except ValueError:
        pass
    else:
        raise AssertionError("應拒絕不合法時間")
if ZONE_DATA_READY:
    winter = build_city_rows(datetime(2026, 1, 15, 12, tzinfo=timezone.utc), cities)
    summer = build_city_rows(datetime(2026, 7, 15, 12, tzinfo=timezone.utc), cities)
    assert find_city(winter, "倫敦")["UTC偏移"] == "UTC+00:00"
    assert find_city(summer, "倫敦")["UTC偏移"] == "UTC+01:00"
    assert find_city(winter, "紐約")["UTC偏移"] == "UTC-05:00"
    assert find_city(summer, "紐約")["UTC偏移"] == "UTC-04:00"
    assert find_city(winter, "雪梨")["UTC偏移"] == "UTC+11:00"
    assert find_city(summer, "雪梨")["UTC偏移"] == "UTC+10:00"
    assert find_city(exercise_rows, "印度")["UTC偏移"] == "UTC+05:30"
    print("城市時區及冬夏日期測試通過。")
else:
    print("尚無時區資料，略過城市時區測試。")
print("固定偏移、跨日、日夜邊界、查詢與停止條件檢查通過。")

本次完成0輪更新。
城市時區及冬夏日期測試通過。
固定偏移、跨日、日夜邊界、查詢與停止條件檢查通過。


## 補充講義來源

以上頁碼對應使用者提供的五份PPTX，投影片署名林英志。函式、串列、受限算式解析、城市時區及整合練習為本次延伸內容，未改寫來源投影片。

來源資料夾：`TQC+ 程式語言第2版入門特訓教材 Python教學影片PPT`。

- Py01_基本認識(1).pptx
- Py01_基本認識(2).pptx
- Py01_基本認識(3).pptx
- Py02_選擇敘述與迴圈(1).pptx
- Py02_選擇敘述與迴圈(2).pptx